# Validation Analysis

This notebook runs all 10 synthetic scenarios through the ColdGuard pipeline and
validates results against expected outcomes.

It also runs the Monte Carlo comparison between MKT and ColdGuard approaches.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from core.utils import parse_csv_log, run_analysis
from core.decision import Decision

DATA_DIR = Path('..') / 'data' / 'raw'

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Run All Scenarios

In [ ]:
SCENARIOS = [
    ('normal_cold_storage.csv',    'DPT'),
    ('brief_excursion_15c.csv',    'DPT'),
    ('extended_excursion_25c.csv', 'DPT'),
    ('multiple_excursions.csv',    'DPT'),
    ('near_freeze.csv',            'DPT'),
    ('logging_gap.csv',            'DPT'),
    ('transport_scenario.csv',     'DPT'),
    ('outreach_scenario.csv',      'OPV'),
    ('freeze_event.csv',           'DPT'),
    ('worst_case.csv',             'OPV'),
]

results = []
for filename, vaccine in SCENARIOS:
    path = DATA_DIR / filename
    if not path.exists():
        print(f'SKIP: {filename} not found')
        continue
    ts, temps = parse_csv_log(path)
    result = run_analysis(vaccine, ts, temps, n_mc_samples=2000)
    d = result['decision_output']
    p = result['posterior_summary']
    results.append({
        'Scenario': filename.replace('.csv', ''),
        'Vaccine': vaccine,
        'Decision': d.decision.value,
        'Mean Potency (%)': f"{p['mean']*100:.1f}",
        '90% CI': f"{p['ci_90_lower']*100:.1f}–{p['ci_90_upper']*100:.1f}",
        'Confidence': f"{d.confidence*100:.0f}%",
        'Warnings': len(result.get('data_quality_warnings', [])),
        'Freeze Events': len(result.get('freeze_events', [])),
    })
    print(f'{filename}: {d.decision.value} ({p["mean"]*100:.1f}%)')

df = pd.DataFrame(results)
print()
print(df.to_string(index=False))

## 2. Potency Distribution Visualization

For key scenarios, show the full Monte Carlo posterior distribution.

In [ ]:
from core.bayesian import monte_carlo_potency_distribution
from core.vaccine_params import VACCINE_DB

scenarios_to_plot = [
    ('normal_cold_storage.csv', 'DPT', 'Normal Storage (DPT)'),
    ('extended_excursion_25c.csv', 'DPT', '24hr @ 25°C (DPT)'),
    ('worst_case.csv', 'OPV', 'Worst Case (OPV)'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (filename, vaccine, title) in zip(axes, scenarios_to_plot):
    path = DATA_DIR / filename
    if not path.exists():
        ax.text(0.5, 0.5, 'File not found', ha='center', va='center', transform=ax.transAxes)
        continue
    ts, temps = parse_csv_log(path)
    vp = VACCINE_DB[vaccine]
    samples = monte_carlo_potency_distribution(
        ts, temps, vp, n_samples=3000, rng=np.random.default_rng(42)
    )
    
    ax.hist(np.array(samples)*100, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(x=vp.min_potency_threshold*100, color='red', linestyle='--',
               label=f'{vp.min_potency_threshold*100:.0f}% threshold')
    ax.set_xlabel('Remaining Potency (%)')
    ax.set_ylabel('Count')
    ax.set_title(title)
    ax.legend()
    
    mean_pot = np.mean(samples) * 100
    ax.axvline(x=mean_pot, color='darkblue', linestyle='-', alpha=0.8, label=f'Mean: {mean_pot:.1f}%')

plt.tight_layout()
plt.show()

## 3. Segment Attribution

Which time segments contributed most to degradation?

In [ ]:
from core.arrhenius import compute_segment_attribution

# Load extended excursion scenario
path = DATA_DIR / 'extended_excursion_25c.csv'
if path.exists():
    ts, temps = parse_csv_log(path)
    vp = VACCINE_DB['DPT']
    
    segs = compute_segment_attribution(ts, temps, vp.Ea_mean, vp.A)
    
    hours = [(s['t_start'] - ts[0]) / 3600 for s in segs]
    contributions = [s['degradation_fraction'] * 100 for s in segs]
    avg_temps = [s['avg_temp_C'] for s in segs]
    
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12, 8))
    
    colors = ['red' if t > 8 else 'steelblue' for t in avg_temps]
    ax1.bar(hours, contributions, color=colors, width=1.0, alpha=0.7)
    ax1.set_ylabel('Degradation Contribution (%)')
    ax1.set_title('DPT — Extended Excursion: Segment Attribution')
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(hours, avg_temps, 'b-', linewidth=2)
    ax2.axhline(y=8, color='red', linestyle='--', alpha=0.5, label='8°C threshold')
    ax2.set_xlabel('Time (hours from start)')
    ax2.set_ylabel('Temperature (°C)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Top degradation hours
    top_segs = sorted(segs, key=lambda s: s['degradation_fraction'], reverse=True)[:5]
    print('Top 5 degradation segments:')
    for s in top_segs:
        h = (s['t_start'] - ts[0]) / 3600
        print(f'  Hour {h:.0f}: {s["avg_temp_C"]:.1f}°C, {s["degradation_fraction"]*100:.2f}% of total degradation')

## 4. Protocol Comparison Summary

Compare VVM, MKT, and ColdGuard protocols across simulated scenarios.

In [ ]:
try:
    from simulation.protocol_comparison import run_comparison_study
    
    print('Running protocol comparison (200 scenarios)...')
    comparison = run_comparison_study(n_scenarios=200, vaccine_type='DPT', seed=42)
    
    print('\nAgreement rates:')
    for pair, rate in comparison.get('agreement_rates', {}).items():
        print(f'  {pair}: {rate*100:.1f}%')
    
    print('\nDecision distributions:')
    for protocol, dist in comparison.get('decision_distributions', {}).items():
        print(f'  {protocol}: {dist}')
except ImportError as e:
    print(f'Simulation module not available: {e}')